<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_06_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 06 — The Report

**Paired with lecture block L6 · Part 1**

The last notebook of Part 1. What is marked is not whether your numbers match
anyone else's — it is whether you can say what you measured, what it means, and
where it stops being true.

## How to use this notebook

Run section 0 to pull in what notebooks 01 to 05 saved. Then replace every
string in section 2. Section 3 assembles the whole thing into a markdown file
you submit.

Word limits are enforced by the check in section 3, and they are tight on
purpose. An answer that needs six hundred words has not been understood yet.

## 0 · Load what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_6_core as core
core.keep_outputs()


In [ ]:
import os
import numpy as np
import Ex_6_core as core

paths = {
    "01 · losses":       "nb01_losses.npz",
    "02 · optimisers":   "nb02_optimisers.npz",
    "03 · transfer":     "nb03_transfer.npz",
    "04 · quantisation": "nb04_quantisation.npz",
    "05 · arbitrage":    "nb05_arbitrage.npz",
}

R = {}
core.needed('nb01_losses.npz', 'nb02_optimisers.npz', 'nb03_transfer.npz', 'nb04_quantisation.npz',
            'nb05_arbitrage.npz')   # on Colab without Drive, asks for the missing files
for label, fname in paths.items():
    full = os.path.join(core.OUTPUT_DIR, fname)
    if os.path.exists(full):
        R[label] = np.load(full, allow_pickle=True)
        print(f"  loaded  {label}")
    else:
        print(f"  MISSING {label}  ({fname}) -- run that notebook first")

if len(R) < len(paths):
    print("\nSome results are missing. The report will have gaps.")

---

## 0b · Your personal seed

Every notebook in this exercise set fixes the seed to 0, so the printed
"what you should see" blocks are true on any machine. That is right for
checking your work and wrong for reporting it — with one seed the whole cohort
produces identical numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints in your answers where the questions ask for it. Your
supervisor can regenerate exactly these numbers from your study number alone,
so they are worth getting right and pointless to invent.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = core.personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own calibration run on the same load cell.
x_you, y_you = core.calibration_dataset(n=40, seed=SEED)
design = np.stack([x_you, np.ones_like(x_you)], axis=1)
A_YOU, B_YOU = np.linalg.lstsq(design, y_you, rcond=None)[0]
SIGMA_YOU = float(np.sqrt(np.mean((y_you - (A_YOU * x_you + B_YOU)) ** 2)))

print()
print(f"  your fitted slope     : {A_YOU:.4f}   (true {core.TRUE_SLOPE})")
print(f"  your fitted intercept : {B_YOU:.4f}   (true {core.TRUE_INTERCEPT})")
print(f"  your residual sigma   : {SIGMA_YOU:.4f}   (true {core.TRUE_SIGMA})")

## 1 · The evidence, gathered

Read these off before writing anything. Several of the questions below are
about numbers you may not have looked at since you produced them.

In [ ]:
if "01 · losses" in R:
    d = R["01 · losses"]
    print("NOTEBOOK 01")
    print(f"  least-squares slope/intercept : {d['a_ls']:.4f} / {d['b_ls']:.4f}")
    print(f"  sigma from residuals          : {d['sigma_hat']:.4f}"
          f"   (true {core.TRUE_SIGMA})")
    print(f"  cross entropy, numpy vs torch : {abs(d['ce_numpy']-d['ce_torch']):.2e}")
    print(f"  accuracy   CE / MSE           : {d['acc_ce']:.3f} / {d['acc_mse']:.3f}")
    print(f"  held-out CE, CE / MSE trained : {d['heldout_ce_ce']:.4f} / "
          f"{d['heldout_ce_mse']:.4f}")
    print(f"  line fit under MSE / MAE      : {d['fit_mse'][0]:.3f} / "
          f"{d['fit_mae'][0]:.3f}   (true {core.TRUE_SLOPE})")

if "02 · optimisers" in R:
    d = R["02 · optimisers"]
    print("\nNOTEBOOK 02")
    for n, v in zip(d["final_names"], d["final_losses"]):
        print(f"  {str(n):<22s} {v:.3e}")

if "03 · transfer" in R:
    d = R["03 · transfer"]
    print("\nNOTEBOOK 03")
    print(f"  source model on A / on B : {d['acc_source_A']:.3f} / "
          f"{d['acc_source_B']:.3f}")
    for n, v in zip(d["strategy_names"], d["strategy_acc"]):
        print(f"  {str(n):<28s} {v:.3f}")

if "04 · quantisation" in R:
    d = R["04 · quantisation"]
    print("\nNOTEBOOK 04")
    print(f"  size  float32 / int8 : {int(d['base_bytes']):,} / "
          f"{int(d['quant_bytes']):,} bytes"
          f"   ({d['base_bytes']/d['quant_bytes']:.2f}x)")
    print(f"  latency ratio        : "
          f"{d['base_seconds']/d['quant_seconds']:.2f}x")
    print(f"  accuracy delta       : "
          f"{d['quant_accuracy']-d['base_accuracy']:+.3f}")

if "05 · arbitrage" in R:
    d = R["05 · arbitrage"]
    print("\nNOTEBOOK 05")
    lp = float(d["lp"].mean())
    for name in ("lp", "rule", "rl", "imitation"):
        v = float(d[name].mean())
        print(f"  {name:<10s} {v:6.2f} EUR per day   ({100 * v / lp:3.0f} % of the optimum)")
    print(f"  linear program / network : {float(d['lp_ms']):.3f} / {float(d['net_ms']):.3f} ms per day")


## 2 · Your answers

Replace every string. Keep to the word limits.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_LOSS_FROM_NOISE = """
(120 words) Notebook 01 showed that least squares is what you get by assuming
Gaussian noise and maximising the likelihood. State the assumption precisely,
then name one measurement situation in your own field where it is wrong, and
say which loss the correct noise assumption would give you instead.
"""

Q2_OPTIMISER_HANDOFF = """
(120 words) Every PINN in Part 2 is trained with Adam and then L-BFGS. Using
your numbers from notebook 02, explain what each contributes. Then state one
thing a very low final loss does NOT establish.
"""

Q3_LEARNING_RATE = """
(100 words) You have two failed training runs. One ends with a loss of nan; the
other converges smoothly to a value far above what the model should reach.
Explain what has gone wrong in each and how you would tell them apart from the
training log alone.
"""

Q4_TRANSFER = """
(150 words) Report the four accuracies from notebook 03 and say which strategy
you would take to the workshop, with reasons. Include the label budget at
which your answer would change, and say how you know.
"""

Q5_DEPLOYMENT = """
(150 words) Report the three deployment numbers from notebook 04. If your
latency did not improve, explain why and do not apologise for it. Then answer
the question the notebook raised: was quantisation the right lever for this
model, or would a narrower float model have been better?
"""

Q6_WHAT_YOU_DISTRUST = """
(120 words) Name the result from Ex_06 you trust least, and say exactly what
experiment would settle it. Answers naming a specific number and a specific
test score higher than general statements about needing more data.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 3 · Check, assemble, save

The check counts words and refuses anything left at its default. Fix what it
reports, then run it again.

In [ ]:
answers = {
    "1 · Where the loss comes from": (Q1_LOSS_FROM_NOISE, 120),
    "2 · Adam, then L-BFGS": (Q2_OPTIMISER_HANDOFF, 120),
    "3 · Two ways a learning rate fails": (Q3_LEARNING_RATE, 100),
    "4 · A second machine, thirty labels": (Q4_TRANSFER, 150),
    "5 · Making it fit on the device": (Q5_DEPLOYMENT, 150),
    "6 · What you distrust": (Q6_WHAT_YOU_DISTRUST, 120),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or "(%d words)" % limit in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = [f"# Ex_06 — Training Lab", "",
             f"**{NAME}** · {GROUP}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    report = "\n".join(lines)

    os.makedirs(core.OUTPUT_DIR, exist_ok=True)
    out = os.path.join(core.OUTPUT_DIR, "Ex06_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write(report)
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 05

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 21, each under its question, to the end of the
report you just wrote. The last one is not from a notebook: it is the
question across all of them, and it concludes the report. On Colab every notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on the lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [ ]:
# notebook-questions v1 -- your answers from notebooks 01 to 05 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · Where the Loss Comes From ----------------------------
    # 01.1 The negative log likelihood and the mean squared error had the same
    # minimiser. Using this as the example, say where a loss function comes
    # from — the steps from choosing a distribution to minimising the negative
    # log likelihood — and give one thing you can do with the first that you
    # cannot do with the second. (-> L6.1 Q1, Q2)
    "01.1": """
""",
    # 01.2 Your fitted $\hat{\sigma}$ came out near the instrument's true
    # value. What distribution does squared error assume? Suppose
    # $\hat{\sigma}$ had come out four times larger: name two different
    # explanations, and say how you would tell them apart. (-> L6.1 Q3)
    "01.2": """
""",
    # 01.3 Both losses gave the same accuracy on the vibration data but
    # different held-out cross entropies. Where does cross entropy come from,
    # and what should an untrained model report here, with three classes — and
    # with ten? Describe an application in which you would not care about the
    # difference, and one in which you would refuse to deploy the
    # worse-calibrated model. (-> L6.1 Q5)
    "01.3": """
""",
    # 01.4 You are given a dataset in which about one reading in fifty is a
    # transcription error, and the rest are Gaussian. Write down the loss you
    # would use and the assumption it corresponds to, and say what it costs you
    # if the noise was Gaussian after all. (-> L6.1 Q4)
    "01.4": """
""",

    # ---- notebook 02 · Four Optimisers, One Problem -------------------------
    # 02.1 Plain gradient descent zig-zags in a narrow valley. Using your SGD
    # and momentum curves, say what momentum changes — and why the target here
    # is noiseless, and what noise would have hidden. (-> L6.1 Q9)
    "02.1": """
""",
    # 02.2 Which failure mode of a badly chosen learning rate is harder to spot
    # in a training log, and why? Answer it for Adam too: what two averages
    # does it keep, what does each fix, and why is its loss not monotone? (->
    # L6.1 Q8, Q9)
    "02.2": """
""",
    # 02.3 Why does L-BFGS need a closure when Adam does not? Say what L-BFGS
    # uses that Adam does not, why it has to be full-batch, and what each call
    # of the closure costs in forward and backward passes. (-> L6.1 Q7, Q9)
    "02.3": """
""",
    # 02.4 Why is cold L-BFGS worse than Adam-then-L-BFGS, given that L-BFGS
    # uses more information per step? When is the hand-off worth it? (-> L6.1
    # Q9)
    "02.4": """
""",

    # ---- notebook 03 · A Second Machine, and Ten Labels Each ----------------
    # 03.1 The machine-A model scored above chance on machine B before any
    # adaptation. What does that tell you about which layers transfer, and what
    # would it have meant if it had scored *exactly* chance? (-> L6.2 Q1, Q2)
    "03.1": """
""",
    # 03.2 Why does fine-tuning at a high learning rate converge towards the
    # from-scratch result rather than towards something worse? (-> L6.2 Q3)
    "03.2": """
""",
    # 03.3 You have thirty labels and a choice between a linear probe and full
    # fine-tuning. What property of the *domain gap* should decide it — when
    # should you unfreeze more than the head? (-> L6.2 Q3, Q4)
    "03.3": """
""",
    # 03.4 In L11 the pretrained body is ResNet-18 and the new data is a few
    # hundred photographs. Which of the numbers in this notebook would you
    # expect to change most, and in which direction? If you had thousands of
    # unlabelled photographs and no labels at all, what task could
    # self-supervised pre-training invent, and what would it buy you? (-> L6.2
    # Q4, Q5)
    "03.4": """
""",

    # ---- notebook 04 · Making It Fit on the Device --------------------------
    # 04.1 Your model is 4x smaller and no less accurate. What would you need
    # to measure before telling a colleague it is 4x faster? (-> L11.1,
    # deployment)
    "04.1": """
""",
    # 04.2 Under what circumstance is quantisation the *wrong* answer to "this
    # model is too big"? Dynamic quantisation needs no calibration data: what
    # does it pay for that convenience, and when would you accept the extra
    # work of static quantisation? (-> L11.1, deployment)
    "04.2": """
""",
    # 04.3 Why did eight bits barely hurt the classifier and badly hurt the
    # regression? Section 6 quantised *weights* only: name one other tensor the
    # deployed network will round, and say which of the two models is more
    # exposed to it. (-> L11.1, deployment)
    "04.3": """
""",
    # 04.4 In L11 a network runs on a Jetson Nano at a frame rate you must
    # measure. Which of this notebook's three numbers is the binding constraint
    # there? At four frames per second on a car at 2 m/s, how far does it
    # travel blind between two predictions, and what are three ways to shorten
    # that? (-> L11.1, deployment)
    "04.4": """
""",

    # ---- notebook 05 · A Battery That Learns to Trade -----------------------
    # 05.1 Name the state, the action and the reward of this battery, and say
    # which of them a control engineer would call the controller and which the
    # cost. Why is the policy judged on the day's profit rather than on each
    # hour's reward? (-> L6.2 Q6, Q7)
    "05.1": """
""",
    # 05.2 The REINFORCE loss multiplies the log-probability of each action by
    # the return that followed, minus the average return. What does one step of
    # gradient descent on it do to an action followed by a better-than-average
    # day, and why does subtracting the average make training less noisy? (->
    # L6.2 Q8)
    "05.2": """
""",
    # 05.3 The linear program earned more than the learned policy, instantly
    # and without training. Why could it, and what does that say about when
    # reinforcement learning is the wrong tool? Name one change to this problem
    # that would make reinforcement learning worth its cost. (-> L6.2 Q9)
    "05.3": """
""",
    # 05.4 The imitator came much closer to the optimum than REINFORCE and runs
    # far faster than the optimiser. What did it need that REINFORCE did not,
    # and what would you expect on a day whose prices look nothing like the
    # training days? (-> L6.2 Q10)
    "05.4": """
""",

    # ---- to conclude, across all the notebooks ------------------------------
    # C Across the notebooks: which one choice — the loss, the optimiser or the
    # starting weights — changed your result most, and what number shows it?
    # (-> L6.1 and L6.2, Exercise slide)
    "C": """
""",
}


In [ ]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'Where the Loss Comes From', 'The negative log likelihood and the mean squared error had the same minimiser. Using this as the example, say where a loss function comes from — the steps from choosing a distribution to minimising the negative log likelihood — and give one thing you can do with the first that you cannot do with the second.', 'L6.1 Q1, Q2'),
    "01.2": ('01', 'Where the Loss Comes From', "Your fitted $\\hat{\\sigma}$ came out near the instrument's true value. What distribution does squared error assume? Suppose $\\hat{\\sigma}$ had come out four times larger: name two different explanations, and say how you would tell them apart.", 'L6.1 Q3'),
    "01.3": ('01', 'Where the Loss Comes From', 'Both losses gave the same accuracy on the vibration data but different held-out cross entropies. Where does cross entropy come from, and what should an untrained model report here, with three classes — and with ten? Describe an application in which you would not care about the difference, and one in which you would refuse to deploy the worse-calibrated model.', 'L6.1 Q5'),
    "01.4": ('01', 'Where the Loss Comes From', 'You are given a dataset in which about one reading in fifty is a transcription error, and the rest are Gaussian. Write down the loss you would use and the assumption it corresponds to, and say what it costs you if the noise was Gaussian after all.', 'L6.1 Q4'),
    "02.1": ('02', 'Four Optimisers, One Problem', 'Plain gradient descent zig-zags in a narrow valley. Using your SGD and momentum curves, say what momentum changes — and why the target here is noiseless, and what noise would have hidden.', 'L6.1 Q9'),
    "02.2": ('02', 'Four Optimisers, One Problem', 'Which failure mode of a badly chosen learning rate is harder to spot in a training log, and why? Answer it for Adam too: what two averages does it keep, what does each fix, and why is its loss not monotone?', 'L6.1 Q8, Q9'),
    "02.3": ('02', 'Four Optimisers, One Problem', 'Why does L-BFGS need a closure when Adam does not? Say what L-BFGS uses that Adam does not, why it has to be full-batch, and what each call of the closure costs in forward and backward passes.', 'L6.1 Q7, Q9'),
    "02.4": ('02', 'Four Optimisers, One Problem', 'Why is cold L-BFGS worse than Adam-then-L-BFGS, given that L-BFGS uses more information per step? When is the hand-off worth it?', 'L6.1 Q9'),
    "03.1": ('03', 'A Second Machine, and Ten Labels Each', 'The machine-A model scored above chance on machine B before any adaptation. What does that tell you about which layers transfer, and what would it have meant if it had scored *exactly* chance?', 'L6.2 Q1, Q2'),
    "03.2": ('03', 'A Second Machine, and Ten Labels Each', 'Why does fine-tuning at a high learning rate converge towards the from-scratch result rather than towards something worse?', 'L6.2 Q3'),
    "03.3": ('03', 'A Second Machine, and Ten Labels Each', 'You have thirty labels and a choice between a linear probe and full fine-tuning. What property of the *domain gap* should decide it — when should you unfreeze more than the head?', 'L6.2 Q3, Q4'),
    "03.4": ('03', 'A Second Machine, and Ten Labels Each', 'In L11 the pretrained body is ResNet-18 and the new data is a few hundred photographs. Which of the numbers in this notebook would you expect to change most, and in which direction? If you had thousands of unlabelled photographs and no labels at all, what task could self-supervised pre-training invent, and what would it buy you?', 'L6.2 Q4, Q5'),
    "04.1": ('04', 'Making It Fit on the Device', 'Your model is 4x smaller and no less accurate. What would you need to measure before telling a colleague it is 4x faster?', 'L11.1, deployment'),
    "04.2": ('04', 'Making It Fit on the Device', 'Under what circumstance is quantisation the *wrong* answer to "this model is too big"? Dynamic quantisation needs no calibration data: what does it pay for that convenience, and when would you accept the extra work of static quantisation?', 'L11.1, deployment'),
    "04.3": ('04', 'Making It Fit on the Device', 'Why did eight bits barely hurt the classifier and badly hurt the regression? Section 6 quantised *weights* only: name one other tensor the deployed network will round, and say which of the two models is more exposed to it.', 'L11.1, deployment'),
    "04.4": ('04', 'Making It Fit on the Device', "In L11 a network runs on a Jetson Nano at a frame rate you must measure. Which of this notebook's three numbers is the binding constraint there? At four frames per second on a car at 2 m/s, how far does it travel blind between two predictions, and what are three ways to shorten that?", 'L11.1, deployment'),
    "05.1": ('05', 'A Battery That Learns to Trade', "Name the state, the action and the reward of this battery, and say which of them a control engineer would call the controller and which the cost. Why is the policy judged on the day's profit rather than on each hour's reward?", 'L6.2 Q6, Q7'),
    "05.2": ('05', 'A Battery That Learns to Trade', 'The REINFORCE loss multiplies the log-probability of each action by the return that followed, minus the average return. What does one step of gradient descent on it do to an action followed by a better-than-average day, and why does subtracting the average make training less noisy?', 'L6.2 Q8'),
    "05.3": ('05', 'A Battery That Learns to Trade', 'The linear program earned more than the learned policy, instantly and without training. Why could it, and what does that say about when reinforcement learning is the wrong tool? Name one change to this problem that would make reinforcement learning worth its cost.', 'L6.2 Q9'),
    "05.4": ('05', 'A Battery That Learns to Trade', 'The imitator came much closer to the optimum than REINFORCE and runs far faster than the optimiser. What did it need that REINFORCE did not, and what would you expect on a day whose prices look nothing like the training days?', 'L6.2 Q10'),
    "C": ('C', 'to conclude', 'Across the notebooks: which one choice — the loss, the optimiser or the starting weights — changed your result most, and what number shows it?', 'L6.1 and L6.2, Exercise slide'),
}

report_md = os.path.join(core.OUTPUT_DIR, "Ex06_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex06_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "",
           "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += ["### To conclude" if nb == "C" else f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "",
                answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))


**What you should see.** `added to .../Ex06_report.md: 21 of 21 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex06_report.md into Ex06_report.pdf, with any figure
# saved as Ex06_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(core.OUTPUT_DIR, "Ex06_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open(os.path.join(core.OUTPUT_DIR, "Ex06_report.md"), encoding="utf-8").read()
figs = sorted(glob.glob("Ex06_report*.png")
              + glob.glob(os.path.join(core.OUTPUT_DIR, "Ex06_report*.png")))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download(pdf_path)
except ImportError:
    pass


## 4 · What Ex_06 was for, and where Part 1 ends

Five claims you can now defend with your own measurements:

* a loss is a noise assumption written down, not a preference;
* an optimiser is chosen for where in the landscape you are, which is why
  Part 2 uses two of them in sequence;
* a trained model transfers to a related problem, and the learning rate decides
  whether it survives the transfer;
* a deployed model is judged on size, latency and accuracy together;
* a problem with a known model should be solved, and a network that imitates
  the solver is fast enough for the loop.

**Part 1 ends here.** From L7 the physics enters the loss. The machinery stays:
the same networks, the same Adam then L-BFGS. And a small residual is still not
a correct answer.